In [1]:
import sys
import json
from pathlib import Path

# Setup root progetto
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_community.embeddings import FastEmbedEmbeddings

from src.config import (
    QDRANT_URL, 
    COLLECTION_NAME, 
    EMBEDDING_MODEL, 
    RETRIEVAL_TOP_K
)
from src.sidecar_manager import SidecarManager
from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget

# Inizializzazione Vector Store con FastEmbed nativo
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
qdrant_client = QdrantClient(url=QDRANT_URL)
vector_store = QdrantVectorStore(
    client=qdrant_client, 
    collection_name=COLLECTION_NAME, 
    embedding=embeddings
)


# Per il DataSet Pre-Taggato
sidecar_path = project_root / "data/processed/test/sidecar_ds1unibase_04_05T.json"

# DataSet privo di Tag
#sidecar_path = project_root / "data/processed/test/sidecar_04_06T.json"

# Inizializzazione sidecar (runnare 1 sola volta altrimenti da il WARNING)
sidecar_mngr = SidecarManager(filepath=sidecar_path)

print("!>> Moduli caricati e connessione a Qdrant/FastEmbed stabilita")

/tmp/ipykernel_292038/3467519055.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings


!>> Moduli caricati e connessione a Qdrant/FastEmbed stabilita


In [2]:
# Caricamento della query di test da eval_queries.json
queries_path = project_root / "data/queries/ds1/eval_queries.json"
with open(queries_path, "r", encoding="utf-8") as f:
    benchmark_queries = json.load(f)

matched = [q for q in benchmark_queries if q.get("id") == "Q_SINGLE_01"]
query_obj = matched[0]
query_text = query_obj["query"]

print(f"?> ID Query: {query_obj['id']}")
print(f"  >>> Testo Query: '{query_text}'\n")

# Vectorizzazione della query
query_vector = embeddings.embed_query(query_text)

# Retrieval vettoriale nativa con query_points
# with_vectors=True in modo da poterli passare al metodo
response = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=RETRIEVAL_TOP_K,
    with_vectors=True,
    with_payload=True
)
raw_records = response.points

print(f"!>> Recuperati {len(raw_records)} record con relativi vettori di embedding!")

?> ID Query: Q_SINGLE_01
  >>> Testo Query: 'Qual è la funzione dei reflection token nell'architettura Self-RAG?'

!>> Recuperati 7 record con relativi vettori di embedding!


In [15]:
builder = KnowledgeGraphBuilder()
tag_overrides = tag_overrides = sidecar_mngr.load_data().get("tag_overrides", {})

### Popolamento dei nodi dai raw_records di Qdrant
for idx, rec in enumerate(raw_records):
    payload = rec.payload or {}
    
    # Costruzione dell'identificativo unico del chunk
    meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload

    # Estrazione sicura con fallback protetti da 'or' (evita il bug del valore None)
    doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"

    raw_idx = payload.get("chunk_index") if payload.get("chunk_index") is not None else meta.get("chunk_index")
    chunk_idx = raw_idx if raw_idx is not None else idx_global

    chunk_id = payload.get("chunk_id") or meta.get("chunk_id") or f"{doc_id}_chunk_{chunk_idx}"
    
    # Estrazione testo
    text = payload.get("text", payload.get("page_content", ""))

    # Estrazione tag - da la priorità a quelli del sidecar, o quelli salvati su Qdrant (in teoria non presenti)
    base_tags = payload.get("user_tags") or meta.get("user_tags") or payload.get("tags") or meta.get("tags") or []
    chunk_sidecar_info = tag_overrides.get(chunk_id, {})
    tags = chunk_sidecar_info.get("user_tags", base_tags)
    
    # Aggiunta del nodo con salvataggio automatico del vettore
    builder.add_chunk_node(
        chunk_id=chunk_id,
        text=text,
        vector=rec.vector,
        tags=tags
    )

### Calcolo automatico degli archi tramite Cosine Similarity (soglia da config)
builder.auto_connect_nodes()

### Generazione del JSON per la visualizzazione
graph_data = builder.to_json_data()

print(f"!>> Grafo generato con successo:")
print(f"   >>> Nodi inseriti: {len(graph_data['nodes'])}")
print(f"   >>> Archi collegati: {len(graph_data['links'])}")

### Inizializzazione e rendering del Widget
widget = ChunkGraphWidget(sidecar=sidecar_mngr)
widget.load_graph(graph_data)

widget

!>> Grafo generato con successo:
   >>> Nodi inseriti: 7
   >>> Archi collegati: 16


In [12]:
from src.tag_assigner import TagAssigner
from src.tag_reranker import TagReranker

tag_assigner = TagAssigner()
tag_reranker = TagReranker(sidecar_manager=widget.sidecar, tag_assigner=tag_assigner)

# Reranking
reranked_results = tag_reranker.rerank(
    query_text=query_text,
    retrieved_points=raw_records
) 

print(f"!>> Reranking completato. Elaborati {len(reranked_results)} chunk candidati.\n")

if reranked_results:
    print(f"  >>> Tag assegnati alla Query: {reranked_results[0].get('query_tags', [])}")
    print("\n--- Dettaglio Punteggi Post-Reranking ---")
    for item in reranked_results:
        print(f"  > ID: {item['chunk_id']}")
        print(f"    Score Qdrant: {item['initial_score']} | Score Finale: {item['final_score']} (Moltiplicatore W_tag: {item['weight_factor']})")
        print(f"    Tag Chunk: {item['chunk_tags']} | Tag Coincidenti: {item['matched_tags']}\n")

!>> Reranking completato. Elaborati 4 chunk candidati.

  >>> Tag assegnati alla Query: ['Vector Retrieval & Embeddings', 'Fine-Tuning & Model Comparison', 'Hierarchical Summarization', 'Evaluation & Self-Correction']

--- Dettaglio Punteggi Post-Reranking ---
  > ID: Self_RAG_chunk_3
    Score Qdrant: 0.5747 | Score Finale: 0.862 (Moltiplicatore W_tag: 1.5)
    Tag Chunk: ['Vector Retrieval & Embeddings', 'Hierarchical Summarization', 'Fine-Tuning & Model Comparison'] | Tag Coincidenti: ['Hierarchical Summarization', 'Fine-Tuning & Model Comparison', 'Vector Retrieval & Embeddings']

  > ID: Self_RAG_chunk_53
    Score Qdrant: 0.6113 | Score Finale: 0.7335 (Moltiplicatore W_tag: 1.2)
    Tag Chunk: ['Vector Retrieval & Embeddings'] | Tag Coincidenti: ['Vector Retrieval & Embeddings']

  > ID: Self_RAG_chunk_7
    Score Qdrant: 0.5749 | Score Finale: 0.6899 (Moltiplicatore W_tag: 1.2)
    Tag Chunk: ['Vector Retrieval & Embeddings', 'Knowledge Graph & Entities'] | Tag Coincidenti: ['Ve

In [13]:
# Ricostruzione del grafo con i metadati di reranking
builder_reranked = KnowledgeGraphBuilder()

for item in reranked_results:
    payload = item.get("payload", {})
    text = payload.get("text", payload.get("page_content", ""))
    
    builder_reranked.add_chunk_node(
        chunk_id=item["chunk_id"],
        text=text,
        vector=item["vector"],
        tags=item["chunk_tags"],
    ) 

builder_reranked.auto_connect_nodes()
graph_data_reranked = builder_reranked.to_json_data()

# DEBUG print
print(f"!>> Grafo aggiornato post-reranking:")
print(f"   >>> Nodi: {len(graph_data_reranked['nodes'])}")
print(f"   >>> Archi: {len(graph_data_reranked['links'])}")

print("\n--- Analisi Impatto Reranking (Score & Moltiplicatori) ---")
for item in reranked_results:
    init_s = item["initial_score"]
    final_s = item["final_score"]
    delta_s = final_s - init_s
    w_factor = item["weight_factor"]
    matched = item["matched_tags"]
    
    # Formattazione per allineare le colonne nei log
    print(
        f"Chunk: {item['chunk_id']:<22} | "
        f"Score: {init_s:.4f} -> {final_s:.4f} (Δ: {delta_s:+.4f}) | "
        f"W_tag: {w_factor:.2f}x | Match: {matched}"
    )
    
# Aggiornamento reattivo istanza widget preesistente
#! NB: è fondamentale usare il widget preesistente e non reistanziarlo
#|     per evitare di avere 2 copie di SidecarManager che puntano allo stesso .json
#!     incombendo così nella problematica delle 2 cache 
widget.load_graph(graph_data_reranked)
widget

!>> Grafo aggiornato post-reranking:
   >>> Nodi: 4
   >>> Archi: 6

--- Analisi Impatto Reranking (Score & Moltiplicatori) ---
Chunk: Self_RAG_chunk_3       | Score: 0.5747 -> 0.8620 (Δ: +0.2873) | W_tag: 1.50x | Match: ['Hierarchical Summarization', 'Fine-Tuning & Model Comparison', 'Vector Retrieval & Embeddings']
Chunk: Self_RAG_chunk_53      | Score: 0.6113 -> 0.7335 (Δ: +0.1222) | W_tag: 1.20x | Match: ['Vector Retrieval & Embeddings']
Chunk: Self_RAG_chunk_7       | Score: 0.5749 -> 0.6899 (Δ: +0.1150) | W_tag: 1.20x | Match: ['Vector Retrieval & Embeddings']
Chunk: Self_RAG_chunk_19      | Score: 0.5748 -> 0.6898 (Δ: +0.1150) | W_tag: 1.20x | Match: ['Vector Retrieval & Embeddings']


In [14]:
from langchain_ollama import ChatOllama
from src.config import (
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD,
    LLM_MODEL,
    OLLAMA_URL,
)

llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_URL,
    keep_alive=DEFAULT_KEEP_ALIVE,
    num_thread=DEFAULT_NUM_THREAD,
    temperature=0,
)

context_blocks = []
for idx, item in enumerate(reranked_results, start=1):
    payload = item.get("payload", {})
    text_content = (
        item.get("text") or payload.get("text") or payload.get("page_content", "")
    )
    chunk_id = item.get("chunk_id")
    score = item.get("final_score", 0.0)

    context_blocks.append(
        f"[CHUNK {idx} | Score Reranked: {score:.4f}]\n{text_content.strip()}"
    )

full_context = "\n\n".join(context_blocks)

prompt = f"""Sei un assistente di ricerca specializzato sugli argomenti del contesto. Rispondi alla domanda seguente basandoti ESCLUSIVAMENTE sul contesto fornito. Se il contesto non contiene informazioni sufficienti o è parziale, evidenzialo chiaramente.

Domanda: {query_text}

Contesto recuperato:
{full_context}

Risposta:"""

# Invocazione del modello
response = llm.invoke(prompt)

# Stampa di Query e Risposta
print("\n" + "=" * 80)
print(f"?>> QUERY: {query_text}")
print("=" * 80)
print("\n   >>> RISPOSTA LLM (Ollama):\n")
print(response.content.strip())
print("\n" + "=" * 80)



?>> QUERY: Qual è la funzione dei reflection token nell'architettura Self-RAG?

   >>> RISPOSTA LLM (Ollama):

I reflection token nell'architettura Self-RAG sono utilizzati per indicare la necessità di ricerca di passaggi aggiuntivi per migliorare la qualità della generazione. Sono categorizzati in retrieval tokens (per indicare la necessità di ricerca) e critique tokens (per indicare la qualità della generazione). Questi token vengono utilizzati per controllare la generazione del modello e per determinare se l'aggiunta di passaggi di ricerca sarebbe utile per migliorare la qualità della generazione.



In [ ]:
"""
Qual è la funzione dei reflection token nell'architettura Self-RAG?
"""

""" Risposta senza noise
Gli reflection token nell'architettura Self-RAG hanno la funzione di indicare la necessità di retrieval e la qualità della generazione. Sono categorizzati in retrieval e critique tokens, dove i retrieval token indicano la necessità di retrieval e i critique token indicano la qualità della generazione. Gli reflection token vengono utilizzati per unificare la generazione con la retrieval e per fornire una valutazione autonoma della generazione.
"""

""" Risposta con noise (aggiungendo tag sbagliati)
I reflection token nell'architettura Self-RAG sono utilizzati per indicare la necessità di ricerca di passaggi aggiuntivi per migliorare la qualità della generazione. Sono categorizzati in retrieval tokens (per indicare la necessità di ricerca) e critique tokens (per indicare la qualità della generazione). Questi token vengono utilizzati per controllare la generazione del modello e per determinare se l'aggiunta di passaggi di ricerca sarebbe utile per migliorare la qualità della generazione.
"""